## Colab setup

Run this cell **first**. It clones the repository, installs dependencies,
copies the raw workbook across from Drive, and points the notebooks at the
clone.

**Put your GitHub token in Colab's secrets panel** (the key icon in the left
sidebar), named `GH_TOKEN`, with "Notebook access" switched on. Do not paste
it into a cell — a pasted token gets pushed to GitHub and GitHub will revoke
it automatically.

Safe to re-run. Outside Colab (local Jupyter) it does nothing, so the same
notebook works in both places.

**Colab wipes `/content` when the runtime disconnects.** Push before you
close the tab, or the run is lost — see the last cell of this notebook.


In [ ]:
# ============================================================
# COLAB SETUP — run first. No-op outside Colab. Safe to re-run.
# ============================================================
import os, sys, subprocess
from pathlib import Path

GH_USER    = "KT-Devv"
GH_REPO    = "student-dropout-prediction-ghana"
GH_BRANCH  = "main"
DRIVE_XLSX = "/content/drive/MyDrive/Ghana_Dropout_Project/ghana_dropout_study_M.xlsx"

GIT_NAME   = "Your Name"          # <-- edit
GIT_EMAIL  = "your@email"         # <-- edit

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if not IN_COLAB:
    print("Not in Colab — skipping setup. Paths resolve from the repo root.")
else:
    def sh(cmd, check=True):
        r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
        if r.stdout.strip(): print(r.stdout.strip()[:2000])
        if check and r.returncode != 0:
            print(r.stderr.strip()[:2000])
        return r

    # ---- token from the secrets panel, never from a pasted string -------
    TOKEN = None
    try:
        from google.colab import userdata
        TOKEN = userdata.get("GH_TOKEN")
    except Exception:
        pass
    if not TOKEN:
        print("No GH_TOKEN secret found. Cloning read-only — you will be able "
              "to run, but NOT push.\n"
              "Add it: key icon in the left sidebar -> GH_TOKEN -> "
              "Notebook access on.")

    # ---- clone (or reuse an existing clone) -----------------------------
    REPO_PATH = Path(f"/content/{GH_REPO}")
    if REPO_PATH.exists():
        print(f"Repo already present at {REPO_PATH} — pulling latest.")
        sh(f"git -C {REPO_PATH} pull --ff-only", check=False)
    else:
        url = (f"https://{TOKEN}@github.com/{GH_USER}/{GH_REPO}.git" if TOKEN
               else f"https://github.com/{GH_USER}/{GH_REPO}.git")
        r = sh(f"git clone -b {GH_BRANCH} {url} {REPO_PATH}", check=False)
        if not REPO_PATH.exists():
            raise RuntimeError(
                "Clone failed. Check GH_USER/GH_REPO/GH_BRANCH above, and that "
                "your GH_TOKEN has Contents: Read and write on this repository."
            )

    os.chdir(REPO_PATH)
    os.environ["DROPOUT_REPO"] = str(REPO_PATH)
    if str(REPO_PATH) not in sys.path:
        sys.path.insert(0, str(REPO_PATH))

    # ---- dependencies ---------------------------------------------------
    if Path("requirements.txt").exists():
        print("installing requirements (quiet, ~1-2 min on a cold runtime)...")
        sh("pip install -q -r requirements.txt", check=False)

    # ---- raw data: the ONLY thing Drive is used for ---------------------
    # Pupil-level data is never committed (ethics: HuSSREC/AP/543/VOL. 5),
    # so it is copied in at runtime and .gitignore keeps it out of git.
    Path("data-raw").mkdir(exist_ok=True)
    target = Path("data-raw") / Path(DRIVE_XLSX).name
    if target.exists():
        print(f"raw workbook already present: {target}")
    else:
        try:
            from google.colab import drive
            if not os.path.exists("/content/drive/MyDrive"):
                drive.mount("/content/drive")
            if os.path.exists(DRIVE_XLSX):
                sh(f'cp "{DRIVE_XLSX}" data-raw/')
                print(f"copied raw workbook -> {target}")
            else:
                print(f"NOT FOUND: {DRIVE_XLSX}\n"
                      "Fix DRIVE_XLSX above, or upload the workbook to "
                      "data-raw/ manually. Notebooks 2-9 don't need it "
                      "(they read data-processed/cleaned_data.csv).")
        except Exception as e:
            print("Drive mount skipped:", e)

    # ---- git identity, needed before any commit -------------------------
    sh(f'git config user.name "{GIT_NAME}"', check=False)
    sh(f'git config user.email "{GIT_EMAIL}"', check=False)

    # ---- the check worth not skipping -----------------------------------
    r = subprocess.run("git status --porcelain", shell=True, text=True,
                       capture_output=True)
    leaked = [l for l in r.stdout.splitlines()
              if "data-raw" in l or "ghana_dropout_study" in l
              or "cleaned_data.csv" in l]
    if leaked:
        print("\n*** WARNING: pupil-level data is NOT being ignored by git ***")
        for l in leaked: print("   ", l)
        print("Do not commit until .gitignore covers these.")
    else:
        print("\ngit is correctly ignoring the raw data.")

    print(f"\nREPO : {os.getcwd()}")
    print(f"push : {'enabled' if TOKEN else 'DISABLED (no GH_TOKEN)'}")


# Notebook 3 — Feature Engineering

## This notebook no longer produces a modelling dataset

That is the single most important change in the whole remediation, so it gets
stated plainly.

GATE-1 failed because R01 Notebook 3 called `scaler.fit_transform(...)` and
`encoder.fit_transform(df[col])` on the whole frame and wrote
`engineered_data.csv`, which Notebook 6b then split. Every encoder, scaler,
median and composite had already seen the held-out rows, so every number in
Tables 3, 4 and 6 came from a leaked pipeline.

**You cannot fix that by patching this notebook.** The defect is the boundary
between this notebook and Notebook 6b: as long as a fitted artefact crosses
that boundary as a CSV, the leak exists. So feature engineering now lives in
`pipeline.preprocess_inside_fold()` and is called inside every fold by every
downstream notebook.

What this notebook does instead:

1. **Documents** the three composites with every governing constant printed,
   which is the Q3 fix (J11 breach — undisclosed constants).
2. **Validates** the pipeline on one fold and asserts the fold-safety
   properties that GATE-1 requires.
3. **Demonstrates** the leak, quantitatively, so M6 can state its size
   instead of asserting it was immaterial.
4. Writes an EDA-only file, clearly named, that no modelling notebook reads.

## The two-composite problem the examination did not catch

R01 Notebook 3 mapped income ordinally (`low:0, medium:1, high:2, hgh:2`).
R01 Notebook 9 — the *corrected* in-fold pipeline — label-encoded it
alphabetically instead. So `socioeconomic_vulnerability_score` had **two
different definitions**, and Table 3/4 and Table 5 were computed on different
features. That is a second, undiagnosed reason the two seed-42 rows disagree,
on top of the leakage fix. Both definitions now come from `config.ORDINAL_MAPS`.

In [ ]:
# ---- bootstrap: repo-relative imports, no drive.mount, no hard-coded path ----
import sys, os
from pathlib import Path

def _find_repo(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p, *p.parents]:
        if (c / "config.py").exists():
            return c
    return p

REPO = Path(os.environ["DROPOUT_REPO"]) if os.environ.get("DROPOUT_REPO") else _find_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

# In Colab, clone the repo first, then run:
#     import os; os.environ["DROPOUT_REPO"] = "/content/student-dropout-prediction-ghana"
# Raw pupil-level data is NOT in the repo (ethics); place it under data-raw/
# locally. Nothing below calls drive.mount().

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from config import *
from pipeline import (preprocess_inside_fold, frozen_split, cv_splits,
                      binarise_target, raw_feature_cols)

banner("NOTEBOOK 3 — FEATURE ENGINEERING (fold-safe)")
OUT = run_dir("notebook03_features")
print("outputs ->", OUT)

df = pd.read_csv(CLEANED_CSV)
train_pool, test_holdout = frozen_split(df)
print(f"train_pool {train_pool.shape}   test_holdout {test_holdout.shape}")

In [ ]:
# ---- 1. THE COMPOSITE SPECIFICATION (Q3 — print every constant) --------
spec = f"""
COMPOSITE FEATURE SPECIFICATION
================================================================

1. attendance_risk_index
   formula : weighted mean over terms of (1 - attendance_fraction)
   where   : attendance_fraction = clip(attendance, 0, {ATTENDANCE_MAX}) / {ATTENDANCE_MAX}
   weights : term_1 = {ATTENDANCE_WEIGHTS[0]}, term_2 = {ATTENDANCE_WEIGHTS[1]}, term_3 = {ATTENDANCE_WEIGHTS[2]}
   rationale : the most recent term carries the greatest weight, because a
             late decline signals imminent departure more strongly than an
             early dip that later corrects. Note this makes the feature an
             operationalisation of the claim that TRAJECTORY matters more than
             LEVEL -- which is Finn's central claim, not Bronfenbrenner's.
   R01 fault : attendance was divided by 100 only when max > 1.0, so values
             above 100% produced a NEGATIVE risk contribution. Now clipped first.

2. socioeconomic_vulnerability_score
   formula : (leap_flag + feeding_flag + income_vulnerability) / 3
   where   : income_vulnerability = 1 - (income_ordinal / {max(ORDINAL_MAPS[SOCIOECONOMIC_COLS['family_income']].values())})
   ordinal : {ORDINAL_MAPS[SOCIOECONOMIC_COLS['family_income']]}
   unknown : 'Unknown' -> NaN + a separate family_income_level_missing flag
   R01 fault : Notebook 9 divided a LABEL-ENCODED income column by its
             training max. Label encoding is alphabetical, so with both
             'High' and the misspelling 'Hgh' present the order was
             (Don't know, High, Hgh, Low, Medium) and the composite was not
             monotone in income. Notebook 3 used a different, ordinal map.
             Two definitions, two sets of results, one feature name.

3. behavioral_engagement_index
   formula : ((1 - warnings_scaled) + participation_scaled + extracurricular_scaled) / 3
   scaling : MinMaxScaler FITTED ON THE TRAINING FOLD ONLY, applied to the
             validation fold. This was the fit_transform-on-everything call
             that GATE-1 cited.
   free text : {STRING_TO_NUMERIC}

ALL THREE are built inside pipeline.preprocess_inside_fold(), step 11.
None is ever computed on the full dataset for modelling purposes.
"""
print(spec)
(OUT / "composite_specification.txt").write_text(spec)
print("Paste this into M7. The R01 M7 described the weights as 'giving greater "
      "weight to the most recent term' without printing them.")

In [ ]:
# ---- 2. VALIDATE fold safety (the GATE-1 assertions) -------------------
splits = cv_splits(train_pool, SPLIT_SEED)
tr, vl = splits[0]
X_tr, y_tr, X_vl, y_vl, meta = preprocess_inside_fold(
    train_pool.iloc[tr], train_pool.iloc[vl], verbose=True)

checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "pass": bool(ok), "detail": detail})
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}  {detail}")

print("\nFOLD-SAFETY CHECKS")
check("validation columns identical to training, same order",
      list(X_vl.columns) == list(X_tr.columns), f"{X_tr.shape[1]} features")
check("no NaN in training fold", X_tr.isna().sum().sum() == 0)
check("no NaN in validation fold", X_vl.isna().sum().sum() == 0)
check("all three composites built", len(meta["composites_present"]) == 3,
      str(meta["composites_present"]))
check("no leakage column survived",
      not any(c in X_tr.columns for c in LEAKAGE_EXACT))
check("school_code handled per config",
      (SCHOOL_COL not in X_tr.columns) if SCHOOL_HANDLING == "drop" else True,
      f"SCHOOL_HANDLING={SCHOOL_HANDLING!r}")
check("attendance within physical bound",
      all(X_tr[c].max() <= ATTENDANCE_MAX for c in ATTENDANCE_COLS if c in X_tr))
check("no train/val row index overlap", len(set(tr) & set(vl)) == 0)

# the decisive one: does the validation fold influence any fitted statistic?
X_tr2, y_tr2, _, _, _ = preprocess_inside_fold(
    train_pool.iloc[tr], train_pool.iloc[vl].sample(frac=0.5, random_state=1))
check("training fold unchanged when the validation fold changes",
      X_tr.shape == X_tr2.shape and np.allclose(
          X_tr.select_dtypes(include=np.number).to_numpy(),
          X_tr2.select_dtypes(include=np.number).to_numpy(), equal_nan=True),
      "<- this is the property GATE-1 requires")

chk = pd.DataFrame(checks)
chk.to_csv(OUT / "fold_safety_checks.csv", index=False)
assert chk["pass"].all(), "fold-safety checks failed; fix before modelling"
print(f"\nall {len(chk)} checks pass")
print(f"features per fold: {meta['n_features']}  "
      f"(raw {len(raw_feature_cols(X_tr))} + composites {len(meta['composites_present'])})")
print("This is the number M2/M7 should state as d, computed per fold.")

In [ ]:
# ---- 3. QUANTIFY THE R01 LEAK (so M6 can state its size) ---------------
# M6 asserted the corrected pipeline left the comparison "materially
# unchanged". Table 5 contradicted that. Measure it instead of asserting it.
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

def preprocess_leaky(df_all, tr_idx, vl_idx):
    """The R01 arrangement: fit everything on the FULL frame, then split."""
    d = df_all.copy()
    d[TARGET] = binarise_target(d[TARGET])
    d = d.drop(columns=[c for c in d.columns if c != TARGET and drop_reason(c)],
               errors="ignore")
    if SCHOOL_HANDLING == "drop" and SCHOOL_COL in d.columns:
        d = d.drop(columns=[SCHOOL_COL])
    y = d[TARGET]; X = d.drop(columns=[TARGET])
    num = X.select_dtypes(include=np.number).columns.tolist()
    cat = [c for c in X.columns if c not in num]
    X[num] = X[num].fillna(X[num].median())                 # full-frame median
    for c in cat:
        m = X[c].mode()
        X[c] = X[c].fillna(m.iloc[0] if len(m) else "unknown")
        X[c] = LabelEncoder().fit_transform(X[c].astype(str))  # full-frame fit
    att = [c for c in ATTENDANCE_COLS if c in X.columns]
    if len(att) == 3:
        m = X[att].to_numpy(dtype=float)
        if np.nanmax(m) > 1.0:
            m = m / 100.0                      # R01: no clipping, so >100 -> negative
        X["attendance_risk_index"] = np.average(1.0 - m, axis=1,
                                                weights=ATTENDANCE_WEIGHTS)
    inc = SOCIOECONOMIC_COLS["family_income"]
    leap = SOCIOECONOMIC_COLS["leap_beneficiary"]
    feed = SOCIOECONOMIC_COLS["school_feeding"]
    if all(c in X.columns for c in (inc, leap, feed)):
        vuln = 1.0 - (X[inc].astype(float) / X[inc].max())   # alphabetical ints
        X["socioeconomic_vulnerability_score"] = (
            X[leap].astype(float) + X[feed].astype(float) + vuln) / 3.0
    b = [BEHAVIOR_COLS[k] for k in ("behavior_warnings", "class_participation",
                                     "extracurricular")]
    if all(c in X.columns for c in b):
        s = MinMaxScaler().fit_transform(                    # full-frame fit
            X[b].apply(pd.to_numeric, errors="coerce").fillna(0).astype(float))
        X["behavioral_engagement_index"] = (
            (1 - s[:, 0]) + s[:, 1] + s[:, 2]) / 3.0
    return (X.iloc[tr_idx], y.iloc[tr_idx].astype(int),
            X.iloc[vl_idx], y.iloc[vl_idx].astype(int))

Xl_tr, yl_tr, Xl_vl, yl_vl = preprocess_leaky(train_pool, tr, vl)

rows = []
for name, a, b in [
    ("attendance_risk_index", X_vl.get("attendance_risk_index"),
     Xl_vl.get("attendance_risk_index")),
    ("socioeconomic_vulnerability_score", X_vl.get("socioeconomic_vulnerability_score"),
     Xl_vl.get("socioeconomic_vulnerability_score")),
    ("behavioral_engagement_index", X_vl.get("behavioral_engagement_index"),
     Xl_vl.get("behavioral_engagement_index")),
]:
    if a is None or b is None:
        continue
    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    rows.append({"composite": name,
                 "in_fold_mean": a.mean(), "leaky_mean": b.mean(),
                 "mean_abs_difference": np.abs(a - b).mean(),
                 "pearson_r": float(np.corrcoef(a, b)[0, 1]),
                 "identical": bool(np.allclose(a, b))})
leak_df = pd.DataFrame(rows)
leak_df.to_csv(OUT / "leak_magnitude_composites.csv", index=False)
print("Composite values: corrected in-fold vs the R01 leaky arrangement")
print(leak_df.round(4).to_string(index=False))
print("\nA pearson_r well below 1.0 on socioeconomic_vulnerability_score is the "
      "income-encoding fault, not the leak -- the two are separate defects that "
      "R01 confounded. Report both in M6, with these numbers.")

In [ ]:
# ---- 4. EDA-only export (read by no modelling notebook) ----------------
X_eda, y_eda, _, _, meta_eda = preprocess_inside_fold(
    train_pool, test_holdout.head(1))     # fitted on the training pool only
eda_out = X_eda.copy(); eda_out[TARGET] = y_eda.to_numpy()
eda_out.to_csv(ENGINEERED_EDA_CSV, index=False)

print(f"wrote {ENGINEERED_EDA_CSV.name}  {eda_out.shape}")
print("\n" + "!"*72)
print("THIS FILE IS FOR FIGURES AND DESCRIPTIVE TABLES ONLY.")
print("No modelling notebook reads it. Feeding it to a model and then")
print("splitting is exactly the R01 arrangement that failed GATE-1.")
print("!"*72)

print("\ncomposite descriptives (training pool, for Table 2):")
print(eda_out[[c for c in COMPOSITES if c in eda_out]]
      .describe().T[["mean", "std", "min", "max"]].round(3).to_string())
print("\ncorrelation with target:")
print(eda_out[[c for c in COMPOSITES if c in eda_out] + [TARGET]]
      .corr()[TARGET].drop(TARGET).round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, c in zip(axes, [c for c in COMPOSITES if c in eda_out]):
    sns.kdeplot(data=eda_out, x=c, hue=TARGET, common_norm=False, ax=ax)
    ax.set_title(c, fontsize=9)
plt.tight_layout(); plt.savefig(OUT / "figures/composite_distributions.png", dpi=200)
plt.close()

write_manifest(OUT, {
    "notebook": "03_features",
    "produces_modelling_dataset": False,
    "n_features_per_fold": int(meta["n_features"]),
    "composites": meta["composites_present"],
    "fold_safety_all_pass": bool(chk["pass"].all()),
    "attendance_weights": ATTENDANCE_WEIGHTS.tolist(),
    "income_ordinal_map": ORDINAL_MAPS.get(SOCIOECONOMIC_COLS["family_income"]),
})
print("\nNEXT: Notebook 4 (baselines, in-fold, training pool only).")

---

## Save your work

Colab wipes `/content` when the runtime disconnects. Run this before you
close the tab — including `results/`, which has to be committed (the previous
review failed partly because the diagnostic CSVs backing the reported tables
were not in the repository).


In [ ]:
# ---- commit and push this run ----
import os, subprocess, sys
if "google.colab" in sys.modules or os.path.exists("/content"):
    MESSAGE = "Notebook 3 Feature Engineering run"     # <-- edit if you like

    def sh(c):
        r = subprocess.run(c, shell=True, text=True, capture_output=True)
        print((r.stdout + r.stderr).strip()[:3000]); return r

    sh("git status --short")
    sh("git add -A")
    sh(f'git commit -m "{MESSAGE}"')
    r = sh("git push")
    if r.returncode != 0:
        print("\nPush failed. Usual causes: no GH_TOKEN secret, or the token "
              "lacks Contents: Read and write. Fix it and re-run this cell — "
              "the commit is already made locally, so nothing is lost until "
              "the runtime disconnects.")
else:
    print("Local run — commit with git as usual.")
